# Task 8 - TF-IDF + SVM

Baseline cổ điển: biến văn bản thành vector TF-IDF rồi phân loại bằng SVM tuyến tính (LinearSVC). Minh hoạ cho cả 3 subtask của Task 8 (StereoQueerEval).

In [1]:
!pip install -q scikit-learn pandas

In [2]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


## Chuẩn bị nhãn cho 3 subtask

In [3]:
# Subtask A - Stereotype Detection (nhị phân)
label2id_A = {"yes": 1, "no": 0}
id2label_A = {v: k for k, v in label2id_A.items()}
data["label_A"] = data["stereotype"].map(label2id_A)
print(label2id_A)
print(id2label_A)
# Subtask B - Hate Speech Classification (3 lớp)
label2id_B = {"no": 0, "yes_implicit": 1, "yes_explicit": 2}
id2label_B = {v: k for k, v in label2id_B.items()}
data["label_B"] = data["hate_speech"].map(label2id_B)
print(label2id_B)
print(id2label_B)
# Subtask C - Target & Identity Identification (multilabel, chỉ trên comment hateful)
def parse_target(x):
    if x == "none":
        return []
    scope, identities = x.split("_", 1)
    return [scope] + identities.split(",")

data["target_labels"] = data["target"].apply(parse_target)
data_c = data[data["hate_speech"] != "no"].reset_index(drop=True)
print(f"Subtask C chỉ dùng {len(data_c)}/{len(data)} comment hateful")

{'yes': 1, 'no': 0}
{1: 'yes', 0: 'no'}
{'no': 0, 'yes_implicit': 1, 'yes_explicit': 2}
{0: 'no', 1: 'yes_implicit', 2: 'yes_explicit'}
Subtask C chỉ dùng 1582/2989 comment hateful


## Chia Train/Test

In [4]:
from sklearn.model_selection import train_test_split

train_A, test_A = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_A"])
train_B, test_B = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_B"])
train_C, test_C = train_test_split(data_c, test_size=0.2, random_state=42)

print(f"A: {len(train_A)}/{len(test_A)}  B: {len(train_B)}/{len(test_B)}  C: {len(train_C)}/{len(test_C)}")

A: 2391/598  B: 2391/598  C: 1265/317


## Hàm dùng chung cho cả 3 subtask

Chỉ cần đổi vectorizer và cờ `multilabel` là dùng lại được cho mọi subtask — minh hoạ cách viết code tái sử dụng thay vì copy-paste 3 lần.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

def run_svm_subtask(train_df, test_df, label_col, vectorizer, target_names=None, multilabel=False):
    X_train = vectorizer.fit_transform(train_df["yt_comment"])
    X_test = vectorizer.transform(test_df["yt_comment"])

    if multilabel:
        mlb = MultiLabelBinarizer()
        y_train = mlb.fit_transform(train_df[label_col])
        y_test = mlb.transform(test_df[label_col])
        print("Các nhãn:", list(mlb.classes_))

        clf = OneVsRestClassifier(LinearSVC(class_weight="balanced"))
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        print(f"F1-micro: {f1_score(y_test, pred, average='micro'):.3f}  "
              f"F1-macro: {f1_score(y_test, pred, average='macro'):.3f}  "
              f"Hamming loss: {hamming_loss(y_test, pred):.3f}")
        return clf, mlb

    clf = LinearSVC(class_weight="balanced")
    clf.fit(X_train, train_df[label_col])
    pred = clf.predict(X_test)
    print(classification_report(test_df[label_col], pred, target_names=target_names))
    return clf, vectorizer

## Subtask A: Stereotype Detection

In [6]:

run_svm_subtask(
    train_A, test_A, "label_A",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_A[0], id2label_A[1]],
)

              precision    recall  f1-score   support

          no       0.73      0.73      0.73       333
         yes       0.66      0.66      0.66       265

    accuracy                           0.70       598
   macro avg       0.70      0.70      0.70       598
weighted avg       0.70      0.70      0.70       598



(LinearSVC(class_weight='balanced'),
 TfidfVectorizer(max_features=20000, min_df=2, ngram_range=(1, 2)))

## Subtask B: Hate Speech Classification

In [7]:
run_svm_subtask(
    train_B, test_B, "label_B",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_B[i] for i in sorted(id2label_B)],
)

              precision    recall  f1-score   support

          no       0.67      0.70      0.68       282
yes_implicit       0.44      0.38      0.41       154
yes_explicit       0.50      0.53      0.52       162

    accuracy                           0.57       598
   macro avg       0.54      0.54      0.54       598
weighted avg       0.57      0.57      0.57       598



(LinearSVC(class_weight='balanced'),
 TfidfVectorizer(max_features=20000, min_df=2, ngram_range=(1, 2)))

In [8]:
print(label2id_B)
print(id2label_B)
print([id2label_B[i] for i in sorted(id2label_B)])

{'no': 0, 'yes_implicit': 1, 'yes_explicit': 2}
{0: 'no', 1: 'yes_implicit', 2: 'yes_explicit'}
['no', 'yes_implicit', 'yes_explicit']


## Subtask C: Target & Identity Identification (multilabel)

Mỗi comment hateful có thể mang nhiều nhãn cùng lúc (ví dụ vừa "group" vừa "l", "g"), nên đây là bài toán multilabel. `MultiLabelBinarizer` biến danh sách nhãn thành vector nhị phân nhiều chiều, còn `OneVsRestClassifier` huấn luyện một bộ phân loại nhị phân riêng cho từng nhãn.

In [9]:
run_svm_subtask(
    train_C, test_C, "target_labels",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    multilabel=True,
)

Các nhãn: ['a', 'b', 'g', 'group', 'i', 'individual', 'l', 'lgbtqia+', 'nb', 'q', 't']
F1-micro: 0.717  F1-macro: 0.349  Hamming loss: 0.108


d:\DataScience\Anaconda\envs\semeval\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


(OneVsRestClassifier(estimator=LinearSVC(class_weight='balanced')),
 MultiLabelBinarizer())